In [3]:
import torch
import torch.nn as nn
from torchvision import models

## Load ResNet model

In [4]:
model = models.resnet18(weights="DEFAULT")
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_sta

In [5]:
model.fc = nn.Linear(
    model.fc.in_features,
    2
)
print(model.fc)

Linear(in_features=512, out_features=2, bias=True)


## Image transformation
### from [1 * 28 * 28] to [3 * 224 * 224]

In [6]:
from torchvision import transforms

resnet_transform = transforms.Compose([
    transforms.Resize((224, 224)),   # Upsampling
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor()
])

## ResNet DataLoaders

In [7]:
from medmnist.dataset import PneumoniaMNIST
from torch.utils.data import DataLoader

train_dataset_resnet = PneumoniaMNIST(
    split="train",
    transform=resnet_transform,
    download=True
)

val_dataset_resnet = PneumoniaMNIST(
    split="val",
    transform=resnet_transform,
    download=True
)

test_dataset_resnet = PneumoniaMNIST(
    split="test",
    transform=resnet_transform,
    download=True
)

In [8]:
train_loader_resnet = DataLoader(
    train_dataset_resnet,
    batch_size=32,
    shuffle=True
)

val_loader_resnet = DataLoader(
    val_dataset_resnet,
    batch_size=32,
    shuffle=False
)

test_loader_resnet = DataLoader(
    test_dataset_resnet,
    batch_size=32,
    shuffle=False
)

In [9]:
images, labels = next(iter(train_loader_resnet))

print("Image shape:", images.shape)
print("Label shape:", labels.shape)

Image shape: torch.Size([32, 3, 224, 224])
Label shape: torch.Size([32, 1])


## Transfer Learning

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)

In [11]:
from tqdm.auto import tqdm

In [12]:
# Train ResNet-18

num_epochs = 10

for epoch in range(num_epochs):
    model.train()

    running_loss = 0.0

    progress_bar = tqdm(
        train_loader_resnet,
        desc=f"Epoch {epoch+1}/{num_epochs}"
    )

    for imgs, labels in train_loader_resnet:
        labels = labels.long().view(-1)

        outputs = model(imgs)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    epoch_loss = running_loss / len(train_loader_resnet)
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {epoch_loss:.4f}"
    )

## Validation

In [16]:
model.eval()

correct_labels = 0
total_labels = 0
running_val_loss = 0.0

with torch.no_grad():

    for images, labels in val_loader_resnet:

        labels = labels.long().view(-1)

        outputs = model(images)

        loss = criterion(outputs, labels)

        running_val_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)


val_loss = running_val_loss / len(val_loader_resnet)

val_accuracy = correct_labels / total_labels * 100

print(f"Val Loss: {val_loss:.4f}")
print(f"Val Accuracy: {val_accuracy:.2f}%")

Val Loss: 0.0687
Val Accuracy: 97.71%


## Test Evaluation

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, roc_auc_score

model.eval()

all_labels = []
all_predictions = []
all_probabilities = []

with torch.no_grad():

    for images, labels in test_loader_resnet:

        labels = labels.long().view(-1)

        outputs = model(images)

        probabilities = torch.softmax(outputs, dim=1)[:, 1]
        predictions = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predictions.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())


# Confusion Matrix
cm = confusion_matrix(all_labels, all_predictions)

print("Confusion Matrix:")
print(cm)


# Accuracy
accuracy = accuracy_score(all_labels, all_predictions)

# TN, FP, FN, TP
TN, FP, FN, TP = cm.ravel()

# Sensitivity
sensitivity = TP / (TP + FN)

# Specificity
specificity = TN / (TN + FP)

# ROC-AUC
roc_auc = roc_auc_score(all_labels, all_probabilities)

print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Sensitivity: {sensitivity * 100:.2f}%")
print(f"Specificity: {specificity * 100:.2f}%")
print(f"ROC-AUC: {roc_auc:.4f}")

Confusion Matrix:
[[138  96]
 [  0 390]]
Accuracy: 84.62%
Sensitivity: 100.00%
Specificity: 58.97%
ROC-AUC: 0.9639


In [ ]:
torch.save(
    model.state_dict(),
    "../results/resnet18_baseline.pth"
)

print("resnet18 model saved successfully.")

resnet18 model saved successfully.


## Calculate class weights

In [25]:
labels = train_dataset_resnet.labels

class_counts = torch.bincount(
    torch.tensor(labels).view(-1)
)

print("Class counts:", class_counts)

Class counts: tensor([1214, 3494])


In [26]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [27]:
class_weights = len(labels) / (2 * class_counts.float())

print("Class weights:", class_weights)

Class weights: tensor([1.9390, 0.6737])


In [28]:
class_weights = class_weights.to(device)

criterion_weighted = nn.CrossEntropyLoss(
    weight=class_weights
)

## Class Imbalance & Threshold Optimization

In [24]:
resnet_model = models.resnet18(weights=None)

# Same final layer used during training
resnet_model.fc = nn.Linear(
    resnet_model.fc.in_features,
    2
)

# Load saved weights
resnet_model.load_state_dict(
    torch.load(
        "../results/resnet18_baseline.pth"
    )
)

# Evaluation mode
resnet_model.eval()

print("Baseline ResNet-18 loaded successfully!")

Baseline ResNet-18 loaded successfully!


In [29]:
weighted_model = models.resnet18(weights="DEFAULT")

weighted_model.fc = nn.Linear(
    weighted_model.fc.in_features,
    2
)

weighted_model = weighted_model.to(device)

In [30]:
optimizer_weighted = torch.optim.Adam(
    weighted_model.parameters(),
    lr=0.0001
)

In [31]:
# Train the weighted model

num_epochs = 10

for epoch in range(num_epochs):

    weighted_model.train()
    running_loss = 0.0

    for images, labels in train_loader_resnet:

        images = images.to(device)
        labels = labels.long().view(-1).to(device)

        outputs = weighted_model(images)

        loss = criterion_weighted(outputs, labels)

        optimizer_weighted.zero_grad()

        loss.backward()

        optimizer_weighted.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader_resnet)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {epoch_loss:.4f}"
    )

Epoch [1/10] Train Loss: 0.1557
Epoch [2/10] Train Loss: 0.0840
Epoch [3/10] Train Loss: 0.0628
Epoch [4/10] Train Loss: 0.0258
Epoch [5/10] Train Loss: 0.0207
Epoch [6/10] Train Loss: 0.0244
Epoch [7/10] Train Loss: 0.0339
Epoch [8/10] Train Loss: 0.0187
Epoch [9/10] Train Loss: 0.0157
Epoch [10/10] Train Loss: 0.0086


In [32]:
torch.save(
    weighted_model.state_dict(),
    "../results/resnet18_weighted.pth"
)

print("resnet18 weighted model saved successfully!")

resnet18 weighted model saved successfully!


## Evaluation of weighted model on validation set

In [33]:
weighted_model.eval()

correct_labels = 0
total_labels = 0
running_val_loss = 0.0

with torch.no_grad():

    for images, labels in val_loader_resnet:

        images = images.to(device)
        labels = labels.long().view(-1).to(device)

        outputs = weighted_model(images)

        loss = criterion_weighted(outputs, labels)

        running_val_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

val_loss_weighted = running_val_loss / len(val_loader_resnet)
val_accuracy_weighted = correct_labels / total_labels * 100

print(f"Weighted Val Loss: {val_loss_weighted:.4f}")
print(f"Weighted Val Accuracy: {val_accuracy_weighted:.2f}%")

Weighted Val Loss: 0.1450
Weighted Val Accuracy: 96.76%


## Evaluation of the weighted model on the test set

In [34]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    roc_auc_score
)

weighted_model.eval()

all_labels = []
all_probabilities = []

with torch.no_grad():

    for images, labels in test_loader_resnet:

        images = images.to(device)
        labels = labels.long().view(-1).to(device)

        outputs = weighted_model(images)

        probabilities = torch.softmax(outputs, dim=1)[:, 1]

        all_labels.extend(labels.cpu().numpy())
        all_probabilities.extend(probabilities.cpu().numpy())


# Convert probabilities to predictions using default threshold = 0.5
all_predictions = [
    1 if p >= 0.5 else 0
    for p in all_probabilities
]


# Confusion matrix
cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("Confusion Matrix:")
print(cm)


# Extract values
tn, fp, fn, tp = cm.ravel()


# Metrics
accuracy = accuracy_score(
    all_labels,
    all_predictions
)

sensitivity = tp / (tp + fn)

specificity = tn / (tn + fp)

roc_auc = roc_auc_score(
    all_labels,
    all_probabilities
)


print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Sensitivity: {sensitivity * 100:.2f}%")
print(f"Specificity: {specificity * 100:.2f}%")
print(f"ROC-AUC: {roc_auc:.4f}")

Confusion Matrix:
[[162  72]
 [  1 389]]
Accuracy: 88.30%
Sensitivity: 99.74%
Specificity: 69.23%
ROC-AUC: 0.9771


## Validation-based threshold selection

In [35]:
weighted_model.eval()

val_labels = []
val_probabilities = []

with torch.no_grad():

    for images, labels in val_loader_resnet:

        images = images.to(device)
        labels = labels.long().view(-1).to(device)

        outputs = weighted_model(images)

        probabilities = torch.softmax(outputs, dim=1)[:, 1]

        val_labels.extend(labels.cpu().numpy())
        val_probabilities.extend(probabilities.cpu().numpy())

In [36]:
import numpy as np

thresholds = np.arange(0.1, 0.91, 0.05)

results = []

for threshold in thresholds:

    predictions = [
        1 if p >= threshold else 0
        for p in val_probabilities
    ]

    cm = confusion_matrix(
        val_labels,
        predictions
    )

    tn, fp, fn, tp = cm.ravel()

    sensitivity = tp / (tp + fn)

    specificity = tn / (tn + fp)

    results.append(
        (
            threshold,
            sensitivity,
            specificity
        )
    )

for threshold, sensitivity, specificity in results:

    print(
        f"Threshold: {threshold:.2f} | "
        f"Sensitivity: {sensitivity*100:.2f}% | "
        f"Specificity: {specificity*100:.2f}%"
    )

Threshold: 0.10 | Sensitivity: 97.94% | Specificity: 88.15%
Threshold: 0.15 | Sensitivity: 97.94% | Specificity: 89.63%
Threshold: 0.20 | Sensitivity: 97.94% | Specificity: 89.63%
Threshold: 0.25 | Sensitivity: 97.94% | Specificity: 91.11%
Threshold: 0.30 | Sensitivity: 97.69% | Specificity: 91.11%
Threshold: 0.35 | Sensitivity: 97.69% | Specificity: 92.59%
Threshold: 0.40 | Sensitivity: 97.69% | Specificity: 92.59%
Threshold: 0.45 | Sensitivity: 97.69% | Specificity: 94.81%
Threshold: 0.50 | Sensitivity: 97.43% | Specificity: 94.81%
Threshold: 0.55 | Sensitivity: 97.43% | Specificity: 94.81%
Threshold: 0.60 | Sensitivity: 97.17% | Specificity: 95.56%
Threshold: 0.65 | Sensitivity: 97.17% | Specificity: 95.56%
Threshold: 0.70 | Sensitivity: 97.17% | Specificity: 95.56%
Threshold: 0.75 | Sensitivity: 97.17% | Specificity: 96.30%
Threshold: 0.80 | Sensitivity: 96.92% | Specificity: 97.04%
Threshold: 0.85 | Sensitivity: 96.92% | Specificity: 97.04%
Threshold: 0.90 | Sensitivity: 96.66% | 

In [37]:
best_threshold = 0.80

print(f"Selected validation threshold: {best_threshold}")

Selected validation threshold: 0.8


In [39]:
# Final test evaluation

test_predictions_threshold = [
    1 if p >= best_threshold else 0
    for p in all_probabilities
]

cm_threshold = confusion_matrix(
    all_labels,
    test_predictions_threshold
)

print("Confusion Matrix:")
print(cm_threshold)

tn, fp, fn, tp = cm_threshold.ravel()

accuracy_threshold = accuracy_score(
    all_labels,
    test_predictions_threshold
)

sensitivity_threshold = tp / (tp + fn)

specificity_threshold = tn / (tn + fp)

roc_auc_threshold = roc_auc_score(
    all_labels,
    all_probabilities
)

print(f"Accuracy: {accuracy_threshold * 100:.2f}%")
print(f"Sensitivity: {sensitivity_threshold * 100:.2f}%")
print(f"Specificity: {specificity_threshold * 100:.2f}%")
print(f"ROC-AUC: {roc_auc_threshold:.4f}")

Confusion Matrix:
[[183  51]
 [  4 386]]
Accuracy: 91.19%
Sensitivity: 98.97%
Specificity: 78.21%
ROC-AUC: 0.9771
